# Turtle Breakout 10 — long only vs long + stop-and-reverse on all 30 coins, risk 1%–100%

Pre-registration: `discussions/Turtle_Breakout/012_claude_tb10_preregistration.md`. Code: `turtle_sizing.py` (0.3.0), `turtle_reverse.py`.

TB 09 on every coin of the TB 04 universe: each coin is its own account, TB 01 longs with or without the TB 07 reverse short, risk $f = 1\%,\dots,100\%$ of the high-water mark per trade; sweeps P (125× cap), C3 (3× cap), EQ (current-equity sizing).
**Decision (binding):** test set = the 29 coins other than BTC (BTC chose the experiment), $f = 2\%$, P sweep, Bonferroni over 3 ($\alpha = 0.0167$ each): H1 shallower max drawdown (sign test), H2 higher growth (sign test), H3 pooled equal-weight sleeves: growth difference lower bound > 0 (block bootstrap). `REVERSE_HELPS` (all three) / `HEDGE_ONLY` (H1) / `NO_IMPROVEMENT`.

Environment: `TB01_DATA_DIR` (default `BINANCE_UM_DIR\long`), `TB10_SYNTHETIC=1`.

In [ ]:
import os, sys, json, gc, time, subprocess, tempfile, warnings, platform
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from dotenv import load_dotenv
warnings.filterwarnings('ignore', category=RuntimeWarning)
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 40); pd.set_option('display.max_rows', 150)
ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'src' / 'data_engineering').is_dir(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src')); load_dotenv(ROOT / '.env')
from data_engineering.modeling import turtle_breakout as tb, turtle_trade_features as tf, turtle_sizing as ts, turtle_reverse as tr

SYNTHETIC = os.getenv('TB10_SYNTHETIC', '0') == '1'
P = tb.Params(); SP = ts.SParams(); RP = tr.RParams()
if SYNTHETIC:
    OUT = Path(tempfile.mkdtemp(prefix='tb10-')) / '10'; SP = ts.SParams(n_boot=500)
    ASSETS = ['BTC', 'ETH', 'SOL', 'XRP', 'BNB', 'DOGE', 'ADA', 'LINK']
    FR = tuple(round(x / 100, 2) for x in (1, 2, 3, 5, 7, 10, 12, 15, 20, 30, 50, 100))
else:
    OUT = ROOT / 'notebooks' / 'Turtle_Breakout' / '10'
    base = Path(os.environ['BINANCE_UM_DIR']) if os.environ.get('BINANCE_UM_DIR') else Path(os.environ['DERIBIT_DATA_ROOT']).parent / 'Binance'
    DATA = Path(os.environ.get('TB01_DATA_DIR', str(base / 'long')))
    ASSETS = list(tf.UNIVERSE); FR = ts.FRACS
OUT.mkdir(parents=True, exist_ok=True)
env = os.environ.copy(); env['PYTHONPATH'] = str(ROOT / 'src') + os.pathsep + env.get('PYTHONPATH', '')
chk = subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-p', 'test_turtle_sizing.py'], cwd=str(ROOT), env=env, capture_output=True, text=True)
print(chk.stderr.strip().splitlines()[-1]); assert chk.returncode == 0
print('coins', len(ASSETS), '| levels', len(FR))

## 1. Sweeps, coin by coin (each coin its own account)

In [ ]:
SWEEPS = (('P', SP.max_lev_primary, 'hwm'), ('C3', SP.max_lev_c3, 'hwm'), ('EQ', SP.max_lev_primary, 'equity'))
KEEP_F = (0.02, 0.05, 0.1)
TABS, CURVES, COV = [], {}, []
T0 = time.time()
for i, a in enumerate(ASSETS):
    t1 = time.time()
    if SYNTHETIC:
        b = tb.synthetic_5m(1600 + 100 * i, seed=200 + i, trend=0.005, start='2020-01-01'); f = tb.synthetic_funding(b, 0.0001)
    else:
        pu = DATA / f'{a}USDT_um_1m.parquet'
        if not pu.exists():
            COV.append({'asset': a, 'status': 'no perp file'}); print(a, 'missing'); continue
        b, _ = tf.load_um(pu)
        fpth = DATA / 'basis' / 'funding' / f'{a}USDT_funding.parquet'
        f = tb.funding_events(pd.read_parquet(fpth)) if fpth.exists() else None
        if a == 'TON':
            b = b[b.index < tr.TON_CUTOFF]
    D = tb.prepare_asset(a, b, f, P)
    start = b.index[0] + pd.Timedelta(days=30)
    L0 = ts.trade_list(D, P, start, RP, with_reverse=False); L1 = ts.trade_list(D, P, start, RP, with_reverse=True)
    for name, cap, sb in SWEEPS:
        for v, L in (('long', L0), ('long_rev', L1)):
            t, c = ts.sweep_trades(D, L, P, start, cap, sb, fracs=FR)
            TABS.append(t.assign(asset=a, sweep=name, variant=v))
            if name == 'P':
                for f_ in KEEP_F:
                    CURVES[(a, v, f_)] = c[f_]
    COV.append({'asset': a, 'status': 'ok', 'set': 'confirm' if a in tf.CONFIRM_ASSETS else 'discovery', 'start': start, 'end': b.index[-1],
                'longs': len(L0), 'reverse_shorts': int((L1.kind == 'reverse').sum()), 'funding': f is not None})
    print(f'{a}: {len(L0)} longs, {int((L1.kind == "reverse").sum())} reverses, {time.time() - t1:.0f}s (total {time.time() - T0:.0f}s)')
    del b, D; gc.collect()
COV = pd.DataFrame(COV); display(COV)
TAB = pd.concat(TABS, ignore_index=True)

## 2. Decision at f = 2% (P sweep), test set = all coins except BTC

In [ ]:
def at(f_, sweep='P'):
    x = TAB[(TAB.sweep == sweep) & (TAB.f.round(4) == round(f_, 4))]
    L = x[x.variant == 'long'].set_index('asset'); R = x[x.variant == 'long_rev'].set_index('asset')
    return pd.DataFrame({'set': COV.set_index('asset').set, 'g_L': L.log_growth_per_year, 'g_LR': R.log_growth_per_year, 'cagr_L': L.cagr, 'cagr_LR': R.cagr,
                         'dd_L': L.max_dd, 'dd_LR': R.max_dd, 'sharpe_L': L.sharpe, 'sharpe_LR': R.sharpe, 'calmar_L': L.calmar, 'calmar_LR': R.calmar,
                         'ruined_L': L.ruined, 'ruined_LR': R.ruined}).dropna(subset=['g_L'])
PC2 = at(0.02); TEST = PC2.drop(index='BTC', errors='ignore')
POOL = {f_: ts.pooled_compare({a: CURVES[(a, 'long', f_)] for a in TEST.index}, {a: CURVES[(a, 'long_rev', f_)] for a in TEST.index}, SP) for f_ in KEEP_F}
DEC = ts.decide_all(TEST, POOL[0.02], SP)
print(json.dumps(DEC, indent=1, default=float))
display(PC2.round(3))
for s in ('confirm', 'discovery'):
    g = TEST[TEST.set == s]
    print(f"{s} (excl. BTC): {len(g)} coins | shallower DD {int((g.dd_LR > g.dd_L).sum())} | higher growth {int((g.g_LR > g.g_L).sum())} | "
          f"median dDD {float((g.dd_LR - g.dd_L).median()):+.3f} | median dgrowth {float((g.g_LR - g.g_L).median()):+.3f}")

## 3. Reported: pooled sleeves at 2/5/10%, every risk level, ruin points

In [ ]:
display(pd.DataFrame(POOL).T.round(4))
rows = []
for f_ in FR:
    x = at(f_).drop(index='BTC', errors='ignore')
    rows.append({'f': f_, 'coins': len(x), 'shallower_dd': int((x.dd_LR > x.dd_L).sum()), 'higher_growth': int((x.g_LR > x.g_L).sum()),
                 'median_dd_L': float(x.dd_L.median()), 'median_dd_LR': float(x.dd_LR.median()),
                 'median_cagr_L': float(x.cagr_L.median()), 'median_cagr_LR': float(x.cagr_LR.median()),
                 'ruined_L': int(x.ruined_L.sum()), 'ruined_LR': int(x.ruined_LR.sum())})
BYF = pd.DataFrame(rows); display(BYF.round(3))
RUIN = []
for (a, sw, v), t in TAB.groupby(['asset', 'sweep', 'variant']):
    RUIN.append({'asset': a, 'sweep': sw, 'variant': v, 'first_ruin_f': float(t.loc[t.ruined, 'f'].min()) if t.ruined.any() else np.nan,
                 'best_f': float(t.loc[t.log_growth_per_year.idxmax(), 'f']), 'max_f_dd_above_-50pct': float(t.loc[t.max_dd > -0.5, 'f'].max()) if (t.max_dd > -0.5).any() else np.nan})
RUIN = pd.DataFrame(RUIN)
display(RUIN.pivot_table(index='asset', columns=['sweep', 'variant'], values=['first_ruin_f', 'max_f_dd_above_-50pct']).round(2))
display(RUIN.groupby(['sweep', 'variant'])[['first_ruin_f', 'best_f', 'max_f_dd_above_-50pct']].median().round(3))

## 4. Figures

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
x = PC2.assign(d_dd=(PC2.dd_LR - PC2.dd_L) * 100, d_g=(PC2.g_LR - PC2.g_L) * 100)
col = np.where(x.index == 'BTC', '#c2571a', np.where(x.set == 'confirm', '#1f5aa6', '0.45'))
axes[0].scatter(x.d_dd, x.d_g, c=col, s=28)
for a, r in x.iterrows():
    axes[0].annotate(a, (r.d_dd, r.d_g), fontsize=7, xytext=(3, 2), textcoords='offset points')
axes[0].axhline(0, color='k', lw=0.6); axes[0].axvline(0, color='k', lw=0.6)
axes[0].set_xlabel('max DD change, pp (right = shallower with reverse)'); axes[0].set_ylabel('log growth change, pp/yr')
axes[0].set_title('Per coin at f = 2% (orange BTC, blue confirmation)', loc='left', fontsize=10)
for f_, al in zip(KEEP_F, (1.0, 0.7, 0.45)):
    for v, c in (('long', '#1f5aa6'), ('long_rev', '#b33f40')):
        e = np.exp(ts.pooled_log_returns({a: CURVES[(a, v, f_)] for a in TEST.index}).cumsum())
        axes[1].plot(e.index, e, color=c, alpha=al, lw=1.2, label=f'{v} f={f_:.0%}')
axes[1].set_yscale('log'); axes[1].legend(frameon=False, fontsize=8, ncol=2); axes[1].grid(alpha=0.25)
axes[1].set_title('Pooled equal-weight sleeves, 29 coins (excl. BTC)', loc='left', fontsize=10)
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(BYF.f * 100, BYF.median_dd_L * 100, color='#1f5aa6', label='median max DD, long')
ax.plot(BYF.f * 100, BYF.median_dd_LR * 100, color='#b33f40', label='median max DD, long + reverse')
ax2 = ax.twinx(); ax2.plot(BYF.f * 100, BYF.ruined_L, color='#1f5aa6', ls=':', label='coins ruined, long'); ax2.plot(BYF.f * 100, BYF.ruined_LR, color='#b33f40', ls=':', label='coins ruined, long + reverse')
ax.set_xlabel('risk per trade f (%)'); ax.set_ylabel('median max DD %'); ax2.set_ylabel('coins ruined')
ax.legend(loc='lower left', frameon=False, fontsize=8); ax2.legend(loc='center right', frameon=False, fontsize=8); ax.grid(alpha=0.25)
plt.tight_layout(); plt.show()

## 5. Save

In [ ]:
TAB.to_csv(OUT / '10_sweeps_all.csv.gz', index=False, compression='gzip')
PC2.to_csv(OUT / '10_per_coin_2pct.csv'); BYF.to_csv(OUT / '10_by_f.csv', index=False); RUIN.to_csv(OUT / '10_ruin.csv', index=False)
COV.to_csv(OUT / '10_coverage.csv', index=False)
pd.DataFrame({f'{a}|{v}|{f_}': c for (a, v, f_), c in CURVES.items()}).to_csv(OUT / '10_curves_P.csv.gz', compression='gzip')
(OUT / '10_decision.json').write_text(json.dumps({'decision': DEC, 'pooled': {str(k): v for k, v in POOL.items()},
    'registration': {'version': ts.VERSION, 'test_set': list(TEST.index), 'f': 0.02, 'sweep': 'P', 'alpha_each': SP.alpha / 3}, 'synthetic': SYNTHETIC}, indent=1, default=str))
git = subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], cwd=str(ROOT), capture_output=True, text=True).stdout.strip()
(OUT / '10_manifest.json').write_text(json.dumps({'version': ts.VERSION, 'tb07_version': tr.VERSION, 'git': git, 'python': platform.python_version(),
    'created_utc': pd.Timestamp.now('UTC').isoformat(), 'assets': ASSETS, 'synthetic': SYNTHETIC}, indent=1))
print('saved to', OUT); print('VERDICT:', DEC['verdict'])